# Track 2 · Stage 1 — Generate the code-mixed **text**

Replication of Biswas et al., Interspeech 2025 (Track 2).

Few-shot-prompt **Gemma 4 26B-A4B** (served by **llama.cpp**) for Hindi-English bigrams →
filter them → expand each into four sentences (~16k). Push the text to
`RohanRamesh/hi-en-synth-cs`.

Audio synthesis is a **separate notebook** (`01b`). It needs `transformers==4.46.1` for
parler-tts; the LLM stage here needs neither that pin nor transformers at all (llama.cpp
uses the GGUF's own chat template). The Hub is the checkpoint between them.

### The model: `unsloth/gemma-4-26B-A4B-it-GGUF`
* **apache-2.0, ungated.** A 25.2B-total / **3.8B-active** MoE, served by **llama.cpp**
  (`backend: llamacpp`). Q4_K_M ≈ 16 GB does **not** fit one T4, so `tensor_split=[0.5, 0.5]`
  spreads the single model across **both** cards.
* *One process, both GPUs.* No cross-GPU sharding here — both T4s hold the one model.
* *Thinking is disabled* (`disable_thinking: true`). The backend tries `enable_thinking=False`
  and strips any reasoning that leaks; the smoke test flags it if thinking survives.
* **Measured** (4,466 calls): ~2 h 26 m wall clock at ~1.1 req/s — much faster than feared,
  because only 3.8B params are active per token.

### What switching to this model did and did NOT fix
Measured against the real MUCS corpus, versus the dense Gemma-4-E4B it replaced:

| | E4B | **26B-A4B** | real MUCS |
|---|---|---|---|
| %Latin | 46.9% | **47.2%** | **25.1%** |
| switch points/sentence | 1.96 | **2.02** | **3.15** |
| filter survival | 37.7% | **85.2%** | 92.3% (paper) |
| unique bigrams | 25,348 | **3,458** | 5,932 (paper) |
| median sentence | 8 words | **12 words** | — |

* **Fixed:** sentence quality. Far more fluent, and filter survival more than doubled to near
  the paper's 92.3%.
* **NOT fixed:** deviation **D11**. The script mix and switch density barely moved — that gap
  appears to come from the *prompt*, not model capacity.
* **New problem:** this model repeats itself, yielding only 3,458 unique bigrams. Hence
  `--n-calls 15000` below rather than the paper's 4,466.

### Before you run
* HF **write** token in Kaggle Secrets as `HF_TOKEN`. Internet **on**, **GPU T4 ×2**.
* Nothing to accept — the model is apache-2.0. (Parler-TTS *is* gated; that bites in `01b`.)

Every LLM call is cached, so a 12 h timeout costs nothing on a re-run.

## 0 · Install

The cu121 **prebuilt** llama-cpp-python wheel (with CUDA offload) — no 15-minute compile. NO transformers/parler-tts here.

In [ ]:
# Prebuilt CUDA wheel: ships with GPU offload, avoids a long source build.
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install -q llama-cpp-python -U --force-reinstall --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich huggingface_hub
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

NOTEBOOK_VERSION = "0.10.1"

import csasr
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}.\n"
    "  package older  -> restart the kernel (Run > Restart & clear); pip skips a\n"
    "                    reinstall when the version looks satisfied.\n"
    "  notebook older -> re-download it from the repo; pip does NOT update .ipynb files."
)

import llama_cpp
print("csasr", csasr.__version__, "| llama_cpp", llama_cpp.__version__,
      "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
assert llama_cpp.llama_supports_gpu_offload(), (
    "llama.cpp has no GPU offload -- the CPU-only wheel got installed. "
    "Re-run the cu121 --extra-index-url line above."
)

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]

REAL_REPO  = "RohanRamesh/mucs-he-cs"
SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
# llama.cpp serves this GGUF split across BOTH T4s (tensor_split). One process,
# both GPUs -- so no per-GPU sharding here (unlike the old dense-model notebook).
LLM  = "unsloth/gemma-4-26B-A4B-it-GGUF"
GGUF = "gemma-4-26B-A4B-it-UD-Q4_K_M.gguf"

WORK  = Path("/kaggle/working")
MAN   = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])   # inherits HF_TOKEN + streams
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed ABOVE "
            f"this traceback - scroll up in this cell's output."
        )

def gen(module, *args):
    """Run one LLM stage. The model spans both GPUs via tensor_split, so this is
    a single subprocess -- NOT sharded. Cache makes it resumable across sessions."""
    run(module, *args, "--backend", "llamacpp", "--model", LLM, "--gguf-file", GGUF)

from csasr.manifest import read_jsonl, write_jsonl

import torch
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}  {free/2**30:5.1f} GiB free / {total/2**30:.1f} GiB")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Pull the in-domain transcripts (few-shot exemplars)

Track 2 never trains on real code-switched audio — we only need the *text* of the MUCS train split.

In [ ]:
from datasets import load_dataset
from csasr.manifest import write_jsonl

train_text = load_dataset(REAL_REPO, "train_text", split="train", token=HF_TOKEN)
write_jsonl(MAN / "mucs_train.jsonl", [dict(r) for r in train_text])
print(f"{len(train_text):,} in-domain sentences for few-shot prompting")
print(train_text[0]["text"])

## 2 · SMOKE TEST — load the model, generate 20 bigrams

Loads the 26B GGUF across both T4s, generates real bigrams, and shows which survive the
script filter — in a few minutes, before committing to the ~8–12 h full run.

**It runs as a subprocess**, so the model is freed cleanly before the real stages start
(Jupyter's `Out[]` history would otherwise pin the VRAM).

Two things to check in the output:
* **thinking is off** — if the smoke test warns that `<think>`/`<|channel|>` markers survived,
  stop and tell the maintainer;
* Gemma often emits **three**-word phrases (`बुनियादी formatting basics`); the filter extracts
  the switch pair from inside them (deviation **D9**), so that is fine.

In [ ]:
run("csasr.llm.smoke", "--backend", "llamacpp", "--model", LLM, "--gguf-file", GGUF,
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--n-calls", "2", "--bigrams-per-call", "10")

## 3 · Generate bigrams

Paper: 44,657 raw → 5,932 unique (13.3%). Single process, both GPUs, cached — safe to re-run after a session timeout.

In [ ]:
gen("csasr.llm.gen_bigrams",
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--out", MAN / "bigrams_raw.jsonl",
    "--cache", CACHE / "bigrams.jsonl",
    "--n-calls", "15000")

## 4 · Filter

Deterministic script filter (one Devanagari token + one Latin token), then an LLM translation check with 3-sample self-consistency.
Paper: 5,932 unique → 5,477 valid (92.3%).

In [ ]:
gen("csasr.llm.filter_bigrams",
    "--raw", MAN / "bigrams_raw.jsonl",
    "--out", MAN / "bigrams_valid.jsonl",
    "--cache", CACHE / "transcheck.jsonl",
    "--items-per-call", "20", "--n-samples", "3")

## 5 · Expand each bigram into four sentences

2 English-matrix, 2 Hindi-matrix. Paper: ~16,000 unique from a theoretical 21,908.

In [ ]:
gen("csasr.llm.gen_sentences",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl",
    "--cache", CACHE / "sentences.jsonl")

### GATE 1 — yields must track the paper

A large divergence in the 13.3% dedup rate means the prompt or temperature is off. Decide here, not after 6 hours of TTS.

In [ ]:
raw   = list(read_jsonl(MAN / "bigrams_raw.jsonl"))
uniq  = {r["bigram"] for r in raw}
valid = list(read_jsonl(MAN / "bigrams_valid.jsonl"))
sents = list(read_jsonl(MAN / "sentences.jsonl"))

rows = [
    ("raw bigrams",    len(raw),   44_657, None),
    ("unique bigrams", len(uniq),   5_932, len(uniq) / max(len(raw), 1)),
    ("valid bigrams",  len(valid),  5_477, len(valid) / max(len(uniq), 1)),
    ("sentences",      len(sents), 16_000, None),
]
print(f"{'metric':<16}{'ours':>10}{'paper':>10}{'survival':>12}")
for name, got, want, surv in rows:
    s = f"{surv:.1%}" if surv else "-"
    print(f"{name:<16}{got:>10,}{want:>10,}{s:>12}")
print("\npaper survival: dedup 13.3%, filter 92.3%")
print("\nGemma 4 26B-A4B is smaller than the paper's 70B (deviation D1), so a lower")
print("valid-bigram yield is expected. What matters is that ENOUGH sentences survive:")
print(f"  -> {len(sents):,} sentences")
print()
est_h = sum(len(r["text"].split()) for r in sents) / 2.4 / 3600
print(f"  projected TTS audio: ~{est_h:.1f} h   (Train_T1 needs 8 h; paper's Train_T2 = 22 h)")
if est_h < 16:
    print("  WARNING: M7 (Train_T2) would train on far less audio than the paper's 22 h,")
    print("           which weakens the M6 -> M7 contrast. Raise --n-calls and re-run;")
    print("           cached calls are free, so only the new ones cost time.")

for r in sents[:5]:
    print("   ", r["text"])

### Repair — strip the matrix-language labels

Gemma prefixes each sentence with its matrix language:

    English: Many software programs have different aliases निर्धारित for commands.

Left in, Parler-TTS would literally **speak** "English colon, many software programs…" and
Whisper would then be **trained to emit `English:`** at the start of every transcript. This
re-cleans the text, re-checks that the bigram survived, and recomputes the matrix language
(the prefix biased it). Seconds — no LLM re-run.

In [ ]:
run("csasr.llm.fix_sentences",
    "--in", MAN / "sentences.jsonl",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl")

sents = list(read_jsonl(MAN / "sentences.jsonl"))
print(f"\n{len(sents):,} sentences ready for TTS:")
for r in sents[:5]:
    print("   ", r["text"])

## 6 · Push the text to the Hub

This is the handoff to `01b`. Push before anything can time out.

In [ ]:
for man, cfg in [("bigrams_valid.jsonl", "bigrams"), ("sentences.jsonl", "sentences")]:
    run("csasr.data.push_to_hub", "--manifest", MAN / man,
        "--repo", SYNTH_REPO, "--config", cfg, "--text-only")
print("\ntext stage complete -> now run 01b_synthesize_audio.ipynb")